In [6]:
#setup
import pandas as pd
from pathlib import Path

DATA = Path("../../data/raw_data")

credits       = pd.read_csv(DATA / "credits.csv")
keywords      = pd.read_csv(DATA / "keywords.csv")
links         = pd.read_csv(DATA / "links.csv")
links_small   = pd.read_csv(DATA / "links_small.csv")
meta          = pd.read_csv(DATA / "movies_metadata.csv", low_memory=False)
ratings_small = pd.read_csv(DATA / "ratings_small.csv")
ratings       = pd.read_csv(
    DATA / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int32"},
)  # smaller dtypes keep 26M rows light on memory

tables = {
    "credits": credits, "keywords": keywords, "links": links,
    "links_small": links_small, "meta": meta,
    "ratings_small": ratings_small, "ratings": ratings,
}

In [7]:
#investigating duplicated rows
for name, df in tables.items():
    print(f"{name:14} rows={len(df):>11,}   fully duplicated rows={df.duplicated().sum():>8,}")

credits        rows=     45,476   fully duplicated rows=      37
keywords       rows=     46,419   fully duplicated rows=     987
links          rows=     45,843   fully duplicated rows=       0
links_small    rows=      9,125   fully duplicated rows=       0
meta           rows=     45,466   fully duplicated rows=      17
ratings_small  rows=    100,004   fully duplicated rows=       0
ratings        rows= 26,024,289   fully duplicated rows=       0


In [8]:
#key tests
def key_report(df, cols, name):
    cols = [cols] if isinstance(cols, str) else cols
    null_rows = df[cols].isna().any(axis=1).sum()
    rows_in_dup_groups = df.duplicated(subset=cols, keep=False).sum()
    extra_copies = df.duplicated(subset=cols).sum()
    print(f"{name:28} key={cols}  null={null_rows:,}  "
          f"rows sharing a key={rows_in_dup_groups:,}  extra copies={extra_copies:,}")

key_report(credits,  "id",                  "credits")
key_report(keywords, "id",                  "keywords")
key_report(meta,     "id",                  "meta")
key_report(links,    "movieId",             "links.movieId")
key_report(links,    "tmdbId",              "links.tmdbId")
key_report(ratings,  ["userId", "movieId"], "ratings")
key_report(ratings_small, ["userId", "movieId"], "ratings_small")

credits                      key=['id']  null=0  rows sharing a key=87  extra copies=44
keywords                     key=['id']  null=0  rows sharing a key=1,972  extra copies=987
meta                         key=['id']  null=0  rows sharing a key=59  extra copies=30
links.movieId                key=['movieId']  null=0  rows sharing a key=0  extra copies=0
links.tmdbId                 key=['tmdbId']  null=219  rows sharing a key=278  extra copies=248
ratings                      key=['userId', 'movieId']  null=0  rows sharing a key=0  extra copies=0
ratings_small                key=['userId', 'movieId']  null=0  rows sharing a key=0  extra copies=0


In [9]:
#repeated ids: exact copies/ conflicting content
def classify_repeated_ids(df, key="id"):
    d = df[df.duplicated(subset=key, keep=False)]
    distinct_per_id = d.drop_duplicates().groupby(key).size()
    print("ids that repeat:                      ", d[key].nunique())
    print("  ...all copies identical:            ", (distinct_per_id == 1).sum())
    print("  ...copies differ in content:        ", (distinct_per_id > 1).sum())
    return d

dup_credits  = classify_repeated_ids(credits)
dup_keywords = classify_repeated_ids(keywords)
dup_meta     = classify_repeated_ids(meta)

ids that repeat:                       43
  ...all copies identical:             36
  ...copies differ in content:         7
ids that repeat:                       985
  ...all copies identical:             985
  ...copies differ in content:         0
ids that repeat:                       29
  ...all copies identical:             16
  ...copies differ in content:         13


In [10]:
#eyeball conflicting case
dup_meta.sort_values("id")[["id", "title", "release_date", "imdb_id", "vote_count"]].head(20)

,id,title,release_date,imdb_id,vote_count
676,105045,The Promise,1995-02-16,tt0111613,1.0
1465,105045,The Promise,1995-02-16,tt0111613,1.0
44821,10991,Pokémon: Spell of the Unknown,2000-07-08,tt0235679,144.0
4114,10991,Pokémon: Spell of the Unknown,2000-07-08,tt0235679,143.0
5710,109962,Rich and Famous,1981-09-23,tt0082992,7.0
20899,109962,Rich and Famous,1981-09-23,tt0082992,7.0
23534,110428,Camille Claudel 1915,2013-03-13,tt2018086,20.0
4356,110428,Camille Claudel 1915,2013-03-13,tt2018086,20.0
24844,11115,Deal,2008-01-29,tt0446676,22.0
14012,11115,Deal,2008-01-29,tt0446676,22.0


In [11]:
#solving NaN puzzle
def nan_puzzle(s, label):
    print(label)
    print("  rows:                         ", len(s))
    print("  NaN:                          ", s.isna().sum())
    print("  nunique():                    ", s.nunique())
    print("  duplicated().sum():           ", s.duplicated().sum())
    print("  duplicated() ignoring NaN:    ", s.dropna().duplicated().sum())

nan_puzzle(meta["imdb_id"], "meta.imdb_id")
nan_puzzle(links["tmdbId"], "links.tmdbId")

meta.imdb_id
  rows:                          45466
  NaN:                           17
  nunique():                     45417
  duplicated().sum():            48
  duplicated() ignoring NaN:     32
links.tmdbId
  rows:                          45843
  NaN:                           219
  nunique():                     45594
  duplicated().sum():            248
  duplicated() ignoring NaN:     30


In [12]:
#hidden missing sort_values
#Empty lists disguised as real values
for name, df, cols in [
    ("credits",  credits,  ["cast", "crew"]),
    ("keywords", keywords, ["keywords"]),
    ("meta",     meta,     ["genres", "production_companies", "production_countries", "spoken_languages"]),
]:
    for c in cols:
        print(f"{name}.{c:22} '[]' count = {(df[c] == '[]').sum():>6,}")

# Zeros that may really mean "unknown"
for c in ["budget", "revenue", "runtime", "vote_count"]:
    num = pd.to_numeric(meta[c], errors="coerce")
    zeros = (num == 0).sum()
    unparseable = num.isna().sum() - meta[c].isna().sum()
    print(f"meta.{c:12} zeros={zeros:>6,}  non-numeric strings={unparseable}")

credits.cast                   '[]' count =  2,418
credits.crew                   '[]' count =    771
keywords.keywords               '[]' count = 14,795
meta.genres                 '[]' count =  2,442
meta.production_companies   '[]' count = 11,875
meta.production_countries   '[]' count =  6,282
meta.spoken_languages       '[]' count =  3,829
meta.budget       zeros=36,573  non-numeric strings=3
meta.revenue      zeros=38,052  non-numeric strings=0
meta.runtime      zeros= 1,558  non-numeric strings=0
meta.vote_count   zeros= 2,899  non-numeric strings=0


In [13]:
#malformed rows in metadata
id_num = pd.to_numeric(meta["id"], errors="coerce")
bad_id = meta[id_num.isna()]
print("rows whose id is not numeric:", len(bad_id))
bad_id.iloc[:, :6]   # look at them with your own eyes

rows whose id is not numeric: 3


,adult,belongs_to_collection,budget,genres,homepage,id
19730,- Written by Ørnås,0.065736,/ff9qCepilowshEtG2GYWwzt2bs4.jpg,"[{'name': 'Carousel Productions', 'id': 11176}...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",1997-08-20
29503,Rune Balot goes to a casino connected to the ...,1.931659,/zV8bHuSL6WXoD6FWogP9j4x80bL.jpg,"[{'name': 'Aniplex', 'id': 2883}, {'name': 'Go...","[{'iso_3166_1': 'US', 'name': 'United States o...",2012-09-29
35587,Avalanche Sharks tells the story of a bikini ...,2.185485,/zaSf5OG7V8X8gqFvly88zDdRm46.jpg,"[{'name': 'Odyssey Media', 'id': 17161}, {'nam...","[{'iso_3166_1': 'CA', 'name': 'Canada'}]",2014-01-01


In [14]:
print(meta["adult"].value_counts(dropna=False))
print(meta["video"].value_counts(dropna=False))
print(meta["status"].value_counts(dropna=False))

bad_adult = meta[~meta["adult"].isin(["True", "False"])]
print("same rows as the bad ids?", set(bad_adult.index) == set(bad_id.index))

adult
False                                                                                                                             45454
True                                                                                                                                  9
 - Written by Ørnås                                                                                                                   1
 Rune Balot goes to a casino connected to the October corporation to try to wrap up her case once and for all.                        1
 Avalanche Sharks tells the story of a bikini contest that turns into a horrifying affair when it is hit by a shark avalanche.        1
Name: count, dtype: int64
video
False    45367
True        93
NaN          6
Name: count, dtype: int64
status
Released           45014
Rumored              230
Post Production       98
NaN                   87
In Production         20
Planned               15
Canceled               2
Name: count, dtype: int64
sam

In [15]:
#missingness patterns
# How many missing values per row?
print(meta.isna().sum(axis=1).value_counts().sort_index())

# Do release_date and status go missing together?
a, b = meta["release_date"].isna(), meta["status"].isna()
print("both:", (a & b).sum(), " only date:", (a & ~b).sum(), " only status:", (~a & b).sum())

# Rows missing the most fields
meta[meta.isna().sum(axis=1) >= 10][["id", "title", "release_date", "status"]].head(20)

0       693
1      5872
2     18705
3     18912
4       966
5       275
6        29
7         8
9         1
10        2
14        1
16        2
Name: count, dtype: int64
both: 4  only date: 83  only status: 83


,id,title,release_date,status
19729,82663,NaN,NaN,NaN
19730,1997-08-20,NaN,1,NaN
29502,122662,NaN,NaN,NaN
29503,2012-09-29,NaN,12,NaN
35586,249260,NaN,NaN,NaN


In [16]:
#ID system and join tests
meta_ids    = set(id_num.dropna().astype(int))
credits_ids = set(credits["id"])
kw_ids      = set(keywords["id"])
link_tmdb   = set(links["tmdbId"].dropna().astype(int))
link_movie  = set(links["movieId"])
rating_ids  = set(ratings["movieId"].unique())

def overlap(a, b, na, nb):
    print(f"{na:16} vs {nb:16} | both={len(a & b):>7,}  only {na}={len(a - b):>7,}  only {nb}={len(b - a):>7,}")

overlap(meta_ids,   credits_ids, "meta.id",       "credits.id")
overlap(meta_ids,   kw_ids,      "meta.id",       "keywords.id")
overlap(credits_ids, kw_ids,     "credits.id",    "keywords.id")
overlap(meta_ids,   link_tmdb,   "meta.id",       "links.tmdbId")
overlap(rating_ids, link_movie,  "ratings.movieId", "links.movieId")
overlap(rating_ids, meta_ids,    "ratings.movieId", "meta.id")   # compare against the line above

meta.id          vs credits.id       | both= 45,432  only meta.id=      1  only credits.id=      0
meta.id          vs keywords.id      | both= 45,432  only meta.id=      1  only keywords.id=      0
credits.id       vs keywords.id      | both= 45,432  only credits.id=      0  only keywords.id=      0
meta.id          vs links.tmdbId     | both= 45,433  only meta.id=      0  only links.tmdbId=    161
ratings.movieId  vs links.movieId    | both= 45,115  only ratings.movieId=      0  only links.movieId=    728
ratings.movieId  vs meta.id          | both=  7,565  only ratings.movieId= 37,550  only meta.id= 37,868


In [17]:
#imdb id format mismatch
print(meta["imdb_id"].dropna().head())
print(links["imdbId"].head())

links["imdb_str"] = "tt" + links["imdbId"].astype(str).str.zfill(7)
overlap(set(meta["imdb_id"].dropna()), set(links["imdb_str"]), "meta.imdb_id", "links.imdb_str")

0    tt0114709
1    tt0113497
2    tt0113228
3    tt0114885
4    tt0113041
Name: imdb_id, dtype: object
0    114709
1    113497
2    113228
3    114885
4    113041
Name: imdbId, dtype: int64
meta.imdb_id     vs links.imdb_str   | both= 45,353  only meta.imdb_id=     64  only links.imdb_str=    490


In [18]:
#are _small file subsets check
print("links_small.movieId ⊆ links.movieId:",
      set(links_small["movieId"]) <= set(links["movieId"]))
print("ratings_small.movieId ⊆ links_small.movieId:",
      set(ratings_small["movieId"]) <= set(links_small["movieId"]))

merged = ratings_small.merge(ratings, how="left",
                             on=["userId", "movieId", "rating", "timestamp"], indicator=True)
print(merged["_merge"].value_counts())

links_small.movieId ⊆ links.movieId: False
ratings_small.movieId ⊆ links_small.movieId: True
_merge
left_only     100004
right_only         0
both               0
Name: count, dtype: int64


In [19]:
#smaller plausibility checks
# Timestamps -> real dates
print(pd.to_datetime(ratings["timestamp"], unit="s").agg(["min", "max"]))

# Odd runtimes
rt = pd.to_numeric(meta["runtime"], errors="coerce")
print(rt.describe())
meta.loc[rt > 600, ["title", "runtime"]]

# Why did Excel say 45,505 credits rows vs pandas' 45,476?
with open(DATA / "credits.csv", encoding="utf-8") as f:
    print("physical lines in file:", sum(1 for _ in f), " vs pandas rows + header:", len(credits) + 1)

min   1995-01-09 11:46:44
max   2017-08-04 06:57:50
Name: timestamp, dtype: datetime64[ns]
count    45203.000000
mean        94.128199
std         38.407810
min          0.000000
25%         85.000000
50%         95.000000
75%        107.000000
max       1256.000000
Name: runtime, dtype: float64
physical lines in file: 45477  vs pandas rows + header: 45477


In [20]:
#making profile reusable
def profile(df):
    out = pd.DataFrame({
        "dtype":   df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique":  df.nunique(),
    })
    return out

for name, df in tables.items():
    print(f"\n## {name}  ({len(df):,} rows)")
    print(profile(df).to_markdown())   # needs: pip install tabulate


## credits  (45,476 rows)
|      | dtype   |   missing |   missing_% |   unique |
|:-----|:--------|----------:|------------:|---------:|
| cast | object  |         0 |           0 |    43019 |
| crew | object  |         0 |           0 |    44669 |
| id   | int64   |         0 |           0 |    45432 |

## keywords  (46,419 rows)
|          | dtype   |   missing |   missing_% |   unique |
|:---------|:--------|----------:|------------:|---------:|
| id       | int64   |         0 |           0 |    45432 |
| keywords | object  |         0 |           0 |    25989 |

## links  (45,843 rows)
|          | dtype   |   missing |   missing_% |   unique |
|:---------|:--------|----------:|------------:|---------:|
| movieId  | int64   |         0 |        0    |    45843 |
| imdbId   | int64   |         0 |        0    |    45843 |
| tmdbId   | float64 |       219 |        0.48 |    45594 |
| imdb_str | object  |         0 |        0    |    45843 |

## links_small  (9,125 rows)
|         

In [21]:
row = links[links["movieId"] == 1]
print(row)

tmdb = int(row["tmdbId"].iloc[0])
meta_ids = pd.to_numeric(meta["id"], errors="coerce")

print(meta.loc[meta_ids == tmdb, ["id", "title"]])   # going through the bridge
print(meta.loc[meta_ids == 1,    ["id", "title"]])   # the shortcut, using 1 directly

   movieId  imdbId  tmdbId   imdb_str
0        1  114709   862.0  tt0114709
    id      title
0  862  Toy Story
Empty DataFrame
Columns: [id, title]
Index: []


In [22]:
bridge = links.dropna(subset=["tmdbId"]).copy()
bridge["tmdbId"] = bridge["tmdbId"].astype(int)
meta_num = meta.assign(id=pd.to_numeric(meta["id"], errors="coerce")).dropna(subset=["id"])
meta_num["id"] = meta_num["id"].astype(int)

rated = pd.DataFrame({"movieId": ratings["movieId"].unique()})
step1 = rated.merge(links[["movieId", "tmdbId"]], on="movieId", how="left")
print("rated movies:                 ", len(rated))
print("lost: no tmdbId in links:     ", step1["tmdbId"].isna().sum())
step2 = step1.dropna(subset=["tmdbId"]).assign(tmdbId=lambda d: d["tmdbId"].astype(int))
print("lost: tmdbId not in meta:     ", (~step2["tmdbId"].isin(meta_num["id"])).sum())

rated movies:                  45115
lost: no tmdbId in links:      213
lost: tmdbId not in meta:      161
